# Chapter 8: Preference Optimization Variants — SimPO, DAPO, and Beyond

> **Book:** Agentic AI Engineering  
> **Part:** 2 — RL Methods  
> **System:** Guardian Angel System (GAS) — Multi-agent diabetic patient supervision

In [ ]:
# Install dependencies
# !pip install numpy matplotlib

In [ ]:
import os
TEST_MODE = os.getenv("NOTEBOOK_TEST_MODE", "0") == "1"
print(f"TEST_MODE: {TEST_MODE}")

---
## 🎯 1. Chapter Goal

By the end of this notebook you will:
- Understand **SimPO** and why length-normalization matters
- Implement **SimPO loss from scratch** without a reference model
- Understand **DAPO's asymmetric clipping** for training stability
- Run an **ablation** comparing DPO vs SimPO on medical preference examples
- Know **when to use which algorithm** via a decision flowchart

---
## 📖 2. Theory Recap

### The DPO Family Tree

DPO spawned a family of variants, each addressing specific limitations:

| Algorithm | Year | Key Innovation | Reference Model? |
|-----------|------|----------------|------------------|
| DPO | 2023 | Closed-form RLHF | ✅ Required |
| SimPO | 2024 | Length normalization, no ref | ❌ Not needed |
| DAPO | 2024 | Asymmetric clipping | ✅ Optional |
| IPO | 2023 | Regularized DPO | ✅ Required |
| KTO | 2024 | Kahneman-Tversky optimization | ✅ Required |

### SimPO: Simple Preference Optimization

SimPO (Meng et al., 2024) makes two key changes to DPO:

1. **No reference model**: Uses average log-probability instead of log-ratio
2. **Length normalization**: Divides by response length to prevent length bias
3. **Margin reward**: Adds a target margin γ to ensure clear separation

$$\mathcal{L}_{\text{SimPO}} = -\log \sigma\left( \frac{\beta}{|y_w|} \log \pi_\theta(y_w|x) - \frac{\beta}{|y_l|} \log \pi_\theta(y_l|x) - \gamma \right)$$

### DAPO: Direct Alignment with Policy Optimization

DAPO introduces **asymmetric clipping** — a higher ceiling for positive advantages:

$$\text{clip}_{\text{DAPO}}(r, A) = \begin{cases} \text{clip}(r, 1-\epsilon_{\text{low}}, 1+\epsilon_{\text{high}}) & \text{if } A > 0 \\ \text{clip}(r, 1-\epsilon_{\text{low}}, 1+\epsilon_{\text{low}}) & \text{if } A \leq 0 \end{cases}$$

Where $\epsilon_{\text{high}} > \epsilon_{\text{low}}$ (e.g., 0.28 vs 0.20).

---
## 🔨 3. Build It

### 3.1 SimPO Loss from Scratch

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def sigmoid(x: float) -> float:
    return 1.0 / (1.0 + np.exp(-np.clip(x, -500, 500)))

def simpo_loss(
    beta: float,
    gamma: float,
    log_pi_chosen: float,
    log_pi_rejected: float,
    len_chosen: int,
    len_rejected: int
) -> tuple:
    """
    SimPO loss: -log(sigmoid((beta/|y_w|)*log_pi(y_w) - (beta/|y_l|)*log_pi(y_l) - gamma))
    
    Key differences from DPO:
    1. No reference model (uses raw log probs, not log-ratios)
    2. Length normalization (divides by token count)
    3. Margin gamma (ensures chosen is clearly better)
    
    Args:
        beta: scaling factor (typically 2.0 for SimPO vs 0.1 for DPO)
        gamma: target margin (typically 0.5)
        log_pi_chosen: sum of log probs for chosen response
        log_pi_rejected: sum of log probs for rejected response
        len_chosen: token length of chosen response
        len_rejected: token length of rejected response
    """
    # Length-normalized log probabilities
    avg_log_pi_chosen = log_pi_chosen / len_chosen
    avg_log_pi_rejected = log_pi_rejected / len_rejected
    
    # Scaled terms
    term_chosen = beta * avg_log_pi_chosen
    term_rejected = beta * avg_log_pi_rejected
    
    # Margin (chosen must beat rejected by at least gamma)
    margin = term_chosen - term_rejected - gamma
    
    # SimPO loss
    loss = -np.log(sigmoid(margin))
    
    return loss, margin, avg_log_pi_chosen, avg_log_pi_rejected

# Example computation
beta = 2.0
gamma = 0.5

# Chosen: longer, more complete response (higher total log prob magnitude)
log_pi_chosen = -45.2   # sum of log probs over all tokens
len_chosen = 20         # 20 tokens

# Rejected: shorter, lower quality response
log_pi_rejected = -38.1  # sum of log probs
len_rejected = 15        # 15 tokens

loss, margin, avg_chosen, avg_rejected = simpo_loss(
    beta, gamma, log_pi_chosen, log_pi_rejected, len_chosen, len_rejected
)

print("=" * 55)
print("SimPO Loss Computation — Step by Step")
print("=" * 55)
print(f"\nInputs:")
print(f"  beta                    = {beta}")
print(f"  gamma (target margin)   = {gamma}")
print(f"  log π(y_w|x)            = {log_pi_chosen} (sum over {len_chosen} tokens)")
print(f"  log π(y_l|x)            = {log_pi_rejected} (sum over {len_rejected} tokens)")
print(f"\nStep 1 — Length normalization:")
print(f"  avg log π(y_w|x) = {log_pi_chosen}/{len_chosen} = {avg_chosen:.4f}")
print(f"  avg log π(y_l|x) = {log_pi_rejected}/{len_rejected} = {avg_rejected:.4f}")
print(f"\nStep 2 — Scaled terms:")
print(f"  β × avg_chosen   = {beta} × {avg_chosen:.4f} = {beta*avg_chosen:.4f}")
print(f"  β × avg_rejected = {beta} × {avg_rejected:.4f} = {beta*avg_rejected:.4f}")
print(f"\nStep 3 — Margin:")
print(f"  margin = {beta*avg_chosen:.4f} - {beta*avg_rejected:.4f} - {gamma} = {margin:.4f}")
print(f"\nStep 4 — SimPO Loss:")
print(f"  loss = -log(sigmoid({margin:.4f})) = {loss:.4f}")
print(f"\n{'✅' if margin > 0 else '❌'} Margin is {'positive' if margin > 0 else 'negative'}: "
      f"model {'correctly prefers' if margin > 0 else 'incorrectly prefers'} chosen")

### 3.2 DPO Loss for Comparison

In [ ]:
def dpo_loss(
    beta: float,
    log_pi_chosen: float,
    log_pi_ref_chosen: float,
    log_pi_rejected: float,
    log_pi_ref_rejected: float
) -> tuple:
    """
    Standard DPO loss using log-ratios.
    Requires a reference model (log_pi_ref_*).
    """
    log_ratio_chosen = log_pi_chosen - log_pi_ref_chosen
    log_ratio_rejected = log_pi_rejected - log_pi_ref_rejected
    margin = beta * (log_ratio_chosen - log_ratio_rejected)
    loss = -np.log(sigmoid(margin))
    return loss, margin

print("DPO vs SimPO: Key Differences")
print("=" * 50)
print()
print("DPO:")
print("  loss = -log(σ(β × (log(π/π_ref)_chosen - log(π/π_ref)_rejected)))")
print("  ✅ Well-tested, widely used")
print("  ❌ Requires reference model (2x memory)")
print("  ❌ Length bias: longer responses get lower log probs")
print()
print("SimPO:")
print("  loss = -log(σ((β/|y_w|)×log_π(y_w) - (β/|y_l|)×log_π(y_l) - γ))")
print("  ✅ No reference model needed (50% less memory)")
print("  ✅ Length-normalized (fair comparison)")
print("  ✅ Margin γ ensures clear preference separation")
print("  ⚠️  Newer, less battle-tested")

### 3.3 DAPO: Asymmetric Clipping

In [ ]:
def ppo_clip(ratio: float, epsilon: float = 0.2) -> float:
    """Standard PPO symmetric clipping."""
    return np.clip(ratio, 1 - epsilon, 1 + epsilon)

def dapo_clip(ratio: float, advantage: float, 
              epsilon_low: float = 0.2, epsilon_high: float = 0.28) -> float:
    """
    DAPO asymmetric clipping.
    For positive advantages: higher ceiling (epsilon_high)
    For negative advantages: symmetric (epsilon_low)
    """
    if advantage > 0:
        # Allow more upside for good responses
        return np.clip(ratio, 1 - epsilon_low, 1 + epsilon_high)
    else:
        # Symmetric for bad responses
        return np.clip(ratio, 1 - epsilon_low, 1 + epsilon_low)

# Visualize clipping regions
ratios = np.linspace(0.5, 1.8, 100)

ppo_clipped = [ppo_clip(r) for r in ratios]
dapo_pos_clipped = [dapo_clip(r, advantage=1.0) for r in ratios]   # positive advantage
dapo_neg_clipped = [dapo_clip(r, advantage=-1.0) for r in ratios]  # negative advantage

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Clipping comparison
ax1.plot(ratios, ratios, 'k--', alpha=0.3, label='No clipping')
ax1.plot(ratios, ppo_clipped, 'b-', linewidth=2, label='PPO (symmetric ε=0.2)')
ax1.plot(ratios, dapo_pos_clipped, 'g-', linewidth=2, label='DAPO positive (ε_high=0.28)')
ax1.plot(ratios, dapo_neg_clipped, 'r-', linewidth=2, label='DAPO negative (ε_low=0.2)')
ax1.axvline(x=1.0, color='gray', linestyle=':', alpha=0.5)
ax1.set_xlabel('Probability Ratio π_θ/π_old')
ax1.set_ylabel('Clipped Ratio')
ax1.set_title('PPO vs DAPO Clipping Regions', fontsize=12, fontweight='bold')
ax1.legend()
ax1.grid(alpha=0.3)

# Plot 2: Difference between DAPO and PPO
diff = np.array(dapo_pos_clipped) - np.array(ppo_clipped)
ax2.fill_between(ratios, diff, 0, where=(diff > 0), alpha=0.5, color='green', label='DAPO allows more')
ax2.fill_between(ratios, diff, 0, where=(diff < 0), alpha=0.5, color='red', label='PPO allows more')
ax2.plot(ratios, diff, 'k-', linewidth=1)
ax2.axhline(y=0, color='black', linewidth=1)
ax2.set_xlabel('Probability Ratio')
ax2.set_ylabel('DAPO - PPO clipped ratio')
ax2.set_title('DAPO vs PPO: Extra Headroom for Positive Advantages', fontsize=12, fontweight='bold')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('dapo_clipping.png', dpi=100, bbox_inches='tight')
plt.show()

print("DAPO key insight: When a response is GOOD (positive advantage),")
print("allow the policy to update MORE aggressively (ε_high=0.28 vs 0.20).")
print("This speeds up learning of good behaviors while keeping safety constraints.")

### 3.4 Ablation: DPO vs SimPO on Medical Preference Examples

In [ ]:
np.random.seed(42)

# 5 medical preference examples with simulated log probabilities
examples = [
    {
        "name": "Hypoglycemia mgmt",
        "len_chosen": 35, "len_rejected": 8,
        "log_pi_chosen": -78.5, "log_pi_rejected": -18.2,
        "log_pi_ref_chosen": -80.1, "log_pi_ref_rejected": -17.8,
    },
    {
        "name": "DKA recognition",
        "len_chosen": 42, "len_rejected": 12,
        "log_pi_chosen": -95.3, "log_pi_rejected": -27.4,
        "log_pi_ref_chosen": -96.0, "log_pi_ref_rejected": -26.9,
    },
    {
        "name": "Insulin dosing",
        "len_chosen": 28, "len_rejected": 10,
        "log_pi_chosen": -62.1, "log_pi_rejected": -22.8,
        "log_pi_ref_chosen": -63.5, "log_pi_ref_rejected": -22.1,
    },
    {
        "name": "HbA1c interpretation",
        "len_chosen": 31, "len_rejected": 9,
        "log_pi_chosen": -70.2, "log_pi_rejected": -20.5,
        "log_pi_ref_chosen": -71.0, "log_pi_ref_rejected": -20.0,
    },
    {
        "name": "Sick day rules",
        "len_chosen": 45, "len_rejected": 11,
        "log_pi_chosen": -101.8, "log_pi_rejected": -24.9,
        "log_pi_ref_chosen": -102.5, "log_pi_ref_rejected": -24.3,
    },
]

# Compute losses for both methods
beta_dpo = 0.1
beta_simpo = 2.0
gamma_simpo = 0.5

results = []
for ex in examples:
    dpo_l, dpo_m = dpo_loss(
        beta_dpo,
        ex['log_pi_chosen'], ex['log_pi_ref_chosen'],
        ex['log_pi_rejected'], ex['log_pi_ref_rejected']
    )
    simpo_l, simpo_m, _, _ = simpo_loss(
        beta_simpo, gamma_simpo,
        ex['log_pi_chosen'], ex['log_pi_rejected'],
        ex['len_chosen'], ex['len_rejected']
    )
    results.append({
        'name': ex['name'],
        'dpo_loss': dpo_l, 'dpo_margin': dpo_m,
        'simpo_loss': simpo_l, 'simpo_margin': simpo_m
    })

# Display results
print("Ablation: DPO vs SimPO on Medical Preference Examples")
print("=" * 70)
print(f"{'Example':<25} {'DPO Loss':>10} {'DPO Margin':>12} {'SimPO Loss':>12} {'SimPO Margin':>14}")
print("-" * 70)
for r in results:
    print(f"{r['name']:<25} {r['dpo_loss']:>10.4f} {r['dpo_margin']:>+12.4f} "
          f"{r['simpo_loss']:>12.4f} {r['simpo_margin']:>+14.4f}")

print(f"\n{'Mean':<25} {np.mean([r['dpo_loss'] for r in results]):>10.4f} "
      f"{np.mean([r['dpo_margin'] for r in results]):>+12.4f} "
      f"{np.mean([r['simpo_loss'] for r in results]):>12.4f} "
      f"{np.mean([r['simpo_margin'] for r in results]):>+14.4f}")

# Visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
names = [r['name'] for r in results]
x = np.arange(len(names))
width = 0.35

ax1.bar(x - width/2, [r['dpo_loss'] for r in results], width, label='DPO', color='#3498db', edgecolor='black')
ax1.bar(x + width/2, [r['simpo_loss'] for r in results], width, label='SimPO', color='#e67e22', edgecolor='black')
ax1.set_title('DPO vs SimPO Loss Comparison', fontsize=12, fontweight='bold')
ax1.set_ylabel('Loss (lower = better)')
ax1.set_xticks(x)
ax1.set_xticklabels(names, rotation=15, ha='right')
ax1.legend()
ax1.grid(axis='y', alpha=0.3)

ax2.bar(x - width/2, [r['dpo_margin'] for r in results], width, label='DPO margin', color='#3498db', edgecolor='black')
ax2.bar(x + width/2, [r['simpo_margin'] for r in results], width, label='SimPO margin', color='#e67e22', edgecolor='black')
ax2.axhline(y=0, color='black', linewidth=1)
ax2.set_title('DPO vs SimPO Reward Margins', fontsize=12, fontweight='bold')
ax2.set_ylabel('Margin (higher = better)')
ax2.set_xticks(x)
ax2.set_xticklabels(names, rotation=15, ha='right')
ax2.legend()
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('dpo_vs_simpo.png', dpi=100, bbox_inches='tight')
plt.show()

### 3.5 Algorithm Selection Flowchart

```
╔══════════════════════════════════════════════════════════════╗
║         PREFERENCE OPTIMIZATION ALGORITHM SELECTOR          ║
╚══════════════════════════════════════════════════════════════╝

START: Do you have a reference model available?
│
├── YES ──→ Is training stability a concern?
│           │
│           ├── YES ──→ DAPO (asymmetric clipping)
│           │           ε_high=0.28, ε_low=0.20
│           │
│           └── NO  ──→ Is length bias a concern?
│                       │
│                       ├── YES ──→ IPO (regularized DPO)
│                       └── NO  ──→ DPO (standard, well-tested)
│
└── NO  ──→ Memory constrained?
            │
            ├── YES ──→ SimPO (no ref model, -50% memory)
            │           β=2.0, γ=0.5
            │
            └── NO  ──→ Need group reasoning?
                        │
                        ├── YES ──→ GRPO (group relative)
                        └── NO  ──→ KTO (Kahneman-Tversky)

GAS Recommendation: SimPO
  → No reference model needed
  → 50% memory savings
  → Length-normalized (fair for varied response lengths)
```

In [ ]:
# Summary comparison table
algorithms = [
    {"name": "DPO",   "ref_model": "Yes", "length_norm": "No",  "margin": "No",  "memory": "2x",   "stability": "High"},
    {"name": "SimPO", "ref_model": "No",  "length_norm": "Yes", "margin": "Yes", "memory": "1x",   "stability": "High"},
    {"name": "DAPO",  "ref_model": "Opt", "length_norm": "No",  "margin": "No",  "memory": "2x",   "stability": "Very High"},
    {"name": "GRPO",  "ref_model": "Yes", "length_norm": "No",  "margin": "No",  "memory": "1.5x", "stability": "High"},
    {"name": "IPO",   "ref_model": "Yes", "length_norm": "No",  "margin": "No",  "memory": "2x",   "stability": "High"},
]

print(f"{'Algorithm':<10} {'Ref Model':>10} {'Length Norm':>12} {'Margin':>8} {'Memory':>8} {'Stability':>12}")
print("-" * 65)
for alg in algorithms:
    marker = " ← GAS" if alg['name'] == 'SimPO' else ""
    print(f"{alg['name']:<10} {alg['ref_model']:>10} {alg['length_norm']:>12} "
          f"{alg['margin']:>8} {alg['memory']:>8} {alg['stability']:>12}{marker}")

---
## 🏥 4. GAS Connection

### SimPO for the Guardian Angel System

The GAS system runs **5 agents concurrently** on limited GPU resources. Memory efficiency is critical.

```
GAS Memory Budget (per agent, 7B model):

DPO approach:
  Policy model:    14 GB (bf16)
  Reference model: 14 GB (bf16)  ← SimPO eliminates this!
  Optimizer:       28 GB
  Total:           56 GB per agent

SimPO approach:
  Policy model:    14 GB (bf16)
  Optimizer:       28 GB
  Total:           42 GB per agent  ← 25% savings!
```

**SimPO is attractive for GAS because it doesn't need a reference model — reducing memory by 50% of model weights.**

Additional benefits for GAS:
1. **Length normalization**: Medical responses vary widely in length — SimPO fairly compares a 10-word response to a 100-word response
2. **Margin γ**: Ensures the model clearly prefers safe advice over unsafe advice (not just marginally)
3. **Simpler pipeline**: Fewer components = easier to audit and maintain in a medical system

---
## ✅ 5. Checkpoint

Test your understanding:

**Q1: Why does SimPO use length normalization?**  
A: Without length normalization, longer responses naturally have lower total log probabilities (more tokens = more uncertainty). Length normalization (dividing by token count) makes the comparison fair regardless of response length.

**Q2: What does the γ (gamma) parameter do in SimPO?**  
A: γ is a target margin that ensures the chosen response must beat the rejected response by at least γ. This prevents the model from learning trivially small preferences and ensures clear separation between safe and unsafe advice.

**Q3: What is DAPO's key innovation over PPO?**  
A: DAPO uses asymmetric clipping — a higher ceiling (ε_high=0.28) for positive advantages and a lower ceiling (ε_low=0.20) for negative advantages. This allows the policy to learn good behaviors more aggressively while maintaining safety constraints.

**Q4: When would you choose DPO over SimPO?**  
A: Choose DPO when: (1) you have a reference model available, (2) you need well-tested behavior, (3) response lengths are similar, or (4) you're following established RLHF pipelines. SimPO is better when memory is constrained or response lengths vary significantly.

**Q5: How much memory does SimPO save compared to DPO for a 7B model in bf16?**  
A: SimPO saves 14 GB (one full copy of the 7B model × 2 bytes/param = 14 GB) by eliminating the reference model. This is roughly 25% of total training memory.